In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_customers_dataset.csv
/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_sellers_dataset.csv
/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_order_reviews_dataset.csv
/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_order_items_dataset.csv
/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_products_dataset.csv
/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_geolocation_dataset.csv
/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/product_category_name_translation.csv
/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_orders_dataset.csv
/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_order_payments_dataset.csv


In [2]:
#Cell 1 Requirements 

!pip install pyspark # Just in case it's not installed
from pyspark.sql import SparkSession

# Start Spark
spark = SparkSession.builder.appName("OlistSafetyProject").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/27 15:50:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
#Cell 2 Data Extraction

import os

# This lists all files available to your Spark session so we can see the real path
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if "products" in filename:
            print(f"Found it! The real path is: {os.path.join(dirname, filename)}")

# PASTE THE PATH YOU COPIED HERE:
path = "/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_products_dataset.csv" 

# Now run the Spark read
df = spark.read.csv(path, header=True, inferSchema=True)

df.printSchema()
df.show(5)

Found it! The real path is: /kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_products_dataset.csv


root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)

+--------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|          product_id|product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+--------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+--------------

In [5]:
#Cell 3 Data Transformation 

# 1. Select specific columns related to size
# (Similar to your lab: data.select(["Name", "Age"]))
selected_df = df.select("product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm")

# 2. Drop Nulls 
# In Cloud DB, we clean 'dirty' data before processing
clean_df = selected_df.na.drop()

# 3. Create 'IsHeavy' column 
# If weight > 3000g, it's '1' (Heavy), else '0' (Light)
# This is a Transformation (Lazy Evaluation)
final_df = clean_df.withColumn("IsHeavy", (clean_df["product_weight_g"] > 3000).cast("integer"))

final_df.show(5)

+----------------+-----------------+-----------------+----------------+-------+
|product_weight_g|product_length_cm|product_height_cm|product_width_cm|IsHeavy|
+----------------+-----------------+-----------------+----------------+-------+
|             225|               16|               10|              14|      0|
|            1000|               30|               18|              20|      0|
|             154|               18|                9|              15|      0|
|             371|               26|                4|              26|      0|
|             625|               20|               17|              13|      0|
+----------------+-----------------+-----------------+----------------+-------+
only showing top 5 rows


In [6]:
#Cell 4

from pyspark.ml.feature import VectorAssembler

# We combine length, height, and width into one column called "features"
assembler = VectorAssembler(
    inputCols=["product_length_cm", "product_height_cm", "product_width_cm"], 
    outputCol="features"
)

# Transform the data to include the features vector
model_ready_data = assembler.transform(final_df)

# Show the result: you will see a new 'features' column at the end
model_ready_data.select("features", "IsHeavy").show(5)

+----------------+-------+
|        features|IsHeavy|
+----------------+-------+
|[16.0,10.0,14.0]|      0|
|[30.0,18.0,20.0]|      0|
| [18.0,9.0,15.0]|      0|
| [26.0,4.0,26.0]|      0|
|[20.0,17.0,13.0]|      0|
+----------------+-------+
only showing top 5 rows


In [7]:
#Cell 5 Training and Predection 

from pyspark.ml.classification import LogisticRegression

# 1. Split data: 70% for training, 30% for testing
train, test = model_ready_data.randomSplit([0.7, 0.3])

# 2. Define the Model
lr = LogisticRegression(labelCol="IsHeavy", featuresCol="features")

# 3. Train the Model (This is an Action - it triggers the computation)
model = lr.fit(train)

# 4. Make Predictions on the test data
predictions = model.transform(test)

# 5. Display the final results
predictions.select("product_weight_g", "features", "rawPrediction", "probability", "prediction").show(10)

+----------------+----------------+--------------------+--------------------+----------+
|product_weight_g|        features|       rawPrediction|         probability|prediction|
+----------------+----------------+--------------------+--------------------+----------+
|               0|[30.0,25.0,30.0]|[1.31305455423190...|[0.78802384443381...|       0.0|
|               2|[11.0,11.0,52.0]|[1.80367596060138...|[0.85859581851471...|       0.0|
|               2|[11.0,11.0,52.0]|[1.80367596060138...|[0.85859581851471...|       0.0|
|              50| [16.0,2.0,11.0]|[5.18151653149855...|[0.99441192423393...|       0.0|
|              50| [16.0,2.0,11.0]|[5.18151653149855...|[0.99441192423393...|       0.0|
|              50| [16.0,2.0,16.0]|[4.83402101779032...|[0.99210830248456...|       0.0|
|              50| [16.0,2.0,23.0]|[4.34752729859879...|[0.98722650648897...|       0.0|
|              50| [16.0,3.0,11.0]|[5.09795977571128...|[0.99392789768182...|       0.0|
|              50| [1